# Soiling Intelligence & Corrective Measures — Solar Power Plant

**Proof of Concept (PoC-2)** | PVDAQ System 9068 | Jan 2024 – Apr 2025

---

**Objective:** Build an end-to-end pipeline that detects performance degradation
caused by panel soiling and generates actionable cleaning recommendations using
environmental sensor data and an XGBoost regression model.

| Module | Description |
|--------|-------------|
| 1 | Library Imports |
| 2 | Data Loading |
| 3 | Cleaning & Preprocessing |
| 4 | Dataset Merging |
| 5 | Feature Engineering |
| 6 | Exploratory Data Analysis |
| 7 | ML Model — XGBoost Regressor |
| 8 | Performance Analysis |
| 9 | Cleaning Recommendation Engine |
| 10 | Operational Dashboard |

## Module 1 — Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9
})

RATED_CAPACITY_KW = 4000  # Nominal plant capacity (kW)

print('All libraries imported successfully.')

## Module 2 — Data Loading

In [ ]:
ac_raw  = pd.read_csv('9068_ac_power_data_20240101_20250430.csv')
irr_raw = pd.read_csv('9068_irradiance_data_20240101_20250430.csv')
env_raw = pd.read_csv('9068_environment_data_20240101_20250430.csv')

print(f'AC Power   : {ac_raw.shape}')
print(f'Irradiance : {irr_raw.shape}')
print(f'Environment: {env_raw.shape}')

In [ ]:
display(ac_raw.head(3))
display(irr_raw.head(3))
display(env_raw.head(3))

## Module 3 — Cleaning & Preprocessing

In [ ]:
def preprocess_ac(df):
    """Clean AC power data and compute total plant power."""
    df = df.copy()
    df['measured_on'] = pd.to_datetime(df['measured_on'])
    df = df.rename(columns={
        'inverter_1_ac_power_(kw)_inv_150143': 'inv1_power',
        'inverter_2_ac_power_(kw)_inv_150144': 'inv2_power'
    })
    df['total_power'] = df['inv1_power'].fillna(0) + df['inv2_power'].fillna(0)
    return df[['measured_on', 'inv1_power', 'inv2_power', 'total_power']]


def preprocess_irr(df):
    """Clean irradiance data — average both pad readings."""
    df = df.copy()
    df['measured_on'] = pd.to_datetime(df['measured_on'])
    pad1 = 'pyranometer_(class_a)_pad_1_poa_irradiance_temp_compensated_(w/m2)_o_149724'
    pad2 = 'pyranometer_(class_a)_pad_2_poa_irradiance_temp_compensated_(w/m2)_o_149726'
    df['irradiance'] = df[[pad1, pad2]].mean(axis=1)
    return df[['measured_on', 'irradiance']]


def preprocess_env(df):
    """Clean environment data — extract key temperature sensors."""
    df = df.copy()
    df['measured_on'] = pd.to_datetime(df['measured_on'])
    df = df.rename(columns={
        'weather_station_ambient_temperature_(c)_o_149727': 'ambient_temp',
        'thermocouple_pad_1_back-of-module_temperature_1_(c)_o_149728': 'module_temp_1',
        'thermocouple_pad_2_back-of-module_temperature_1_(c)_o_149730': 'module_temp_2'
    })
    df['module_temp'] = df[['module_temp_1', 'module_temp_2']].mean(axis=1)
    return df[['measured_on', 'ambient_temp', 'module_temp']]


ac  = preprocess_ac(ac_raw)
irr = preprocess_irr(irr_raw)
env = preprocess_env(env_raw)

print('Preprocessing complete.')
print(f'AC  : {ac.shape}  |  Irr : {irr.shape}  |  Env : {env.shape}')

## Module 4 — Dataset Merging

In [ ]:
merged = (
    ac
    .merge(irr, on='measured_on', how='inner')
    .merge(env, on='measured_on', how='inner')
)

merged = merged.dropna()
merged = merged.sort_values('measured_on').reset_index(drop=True)

print(f'Merged dataset shape : {merged.shape}')
print(f'Date range           : {merged["measured_on"].min()} → {merged["measured_on"].max()}')
print(f'Missing values       : {merged.isnull().sum().sum()}')
display(merged.head())

## Module 5 — Feature Engineering

In [ ]:
def engineer_features(df, rated_kw=RATED_CAPACITY_KW):
    """Create domain-specific features for soiling analysis."""
    df = df.copy()

    # Performance Ratio (actual / rated × normalised irradiance)
    df['performance_ratio'] = np.where(
        df['irradiance'] > 50,
        df['total_power'] / (rated_kw * df['irradiance'] / 1000),
        np.nan
    )

    # Estimated Power Loss
    df['est_power_loss'] = np.where(
        df['irradiance'] > 50,
        (rated_kw * df['irradiance'] / 1000) - df['total_power'],
        0
    )
    df['est_power_loss'] = df['est_power_loss'].clip(lower=0)

    # High Irradiance Flag (> 800 W/m²)
    df['high_irradiance_flag'] = (df['irradiance'] > 800).astype(int)

    # Rolling Average Power (12-step window = 1 hour at 5-min intervals)
    df['rolling_avg_power'] = df['total_power'].rolling(window=12, min_periods=1).mean()

    # Rolling Performance Ratio (12-step window)
    df['rolling_perf_ratio'] = df['performance_ratio'].rolling(window=12, min_periods=1).mean()

    # Temperature Difference (module - ambient)
    df['temp_diff'] = df['module_temp'] - df['ambient_temp']

    # Daily Energy (kWh) — sum of power readings × 5 min / 60
    df['date'] = df['measured_on'].dt.date
    daily_energy = (
        df.groupby('date')['total_power']
        .sum()
        .mul(5 / 60)
        .rename('daily_energy_kwh')
    )
    df = df.merge(daily_energy, on='date', how='left')

    return df


merged = engineer_features(merged)

print('Engineered features:')
new_cols = [
    'performance_ratio', 'est_power_loss', 'high_irradiance_flag',
    'rolling_avg_power', 'rolling_perf_ratio', 'temp_diff', 'daily_energy_kwh'
]
display(merged[new_cols].describe().round(2))

## Module 6 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Irradiance vs Power
daytime = merged[merged['irradiance'] > 50]
axes[0].scatter(
    daytime['irradiance'], daytime['total_power'],
    alpha=0.15, s=8, color='#2196F3'
)
axes[0].set_xlabel('Irradiance (W/m²)')
axes[0].set_ylabel('Total Power (kW)')
axes[0].set_title('Irradiance vs AC Power Output')

# 2. Daily Power Trend
daily = merged.groupby('date')['total_power'].mean()
axes[1].plot(daily.index, daily.values, linewidth=0.8, color='#FF5722')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Avg Power (kW)')
axes[1].set_title('Daily Average Power Trend')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
corr_cols = [
    'total_power', 'irradiance', 'ambient_temp', 'module_temp',
    'temp_diff', 'est_power_loss', 'rolling_avg_power'
]
corr_matrix = merged[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, ax=ax, vmin=-1, vmax=1,
    annot_kws={'size': 9}
)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
daytime_pr = merged.loc[
    (merged['irradiance'] > 200) & (merged['performance_ratio'].notna()),
    'performance_ratio'
].clip(0, 1.5)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(daytime_pr, bins=80, color='#4CAF50', edgecolor='white', alpha=0.85)
ax.axvline(x=1.0, color='red', linestyle='--', label='PR = 1.0 (Ideal)')
ax.axvline(x=0.8, color='orange', linestyle='--', label='PR = 0.8 (Threshold)')
ax.set_xlabel('Performance Ratio')
ax.set_ylabel('Frequency')
ax.set_title('Performance Ratio Distribution (Daytime Only)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sample = daytime.sample(min(5000, len(daytime)), random_state=42)
sc = ax.scatter(
    sample['temp_diff'], sample['total_power'],
    c=sample['irradiance'], cmap='plasma', alpha=0.5, s=10
)
plt.colorbar(sc, label='Irradiance (W/m²)')
ax.set_xlabel('Module − Ambient Temperature (°C)')
ax.set_ylabel('Total Power (kW)')
ax.set_title('Temperature Difference vs Power (coloured by Irradiance)')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
daily_pr = merged.loc[
    merged['performance_ratio'].notna()
].groupby('date')['performance_ratio'].median()

ax.plot(daily_pr.index, daily_pr.values, linewidth=0.8, color='#673AB7')
ax.axhline(y=1.0, color='green', linestyle='--', alpha=0.6, label='Ideal PR')
ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.6, label='Soiling Threshold')
ax.set_xlabel('Date')
ax.set_ylabel('Median Performance Ratio')
ax.set_title('Daily Median Performance Ratio Trend')
ax.legend()
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Module 7 — XGBoost Regressor Model

The model learns the relationship between environmental conditions and power
output **under normal operation**. Its predictions serve as the *expected*
baseline — any gap between expected and actual power signals potential soiling.

In [ ]:
features = ['irradiance', 'ambient_temp', 'module_temp', 'temp_diff']
target   = 'total_power'

model_df = merged.dropna(subset=features + [target]).copy()

X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Training samples : {len(X_train):,}')
print(f'Testing samples  : {len(X_test):,}')

In [ ]:
model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print('Model training complete.')

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('Model Evaluation')
print(f'  MAE  : {mae:.2f} kW')
print(f'  RMSE : {rmse:.2f} kW')
print(f'  R²   : {r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted scatter
axes[0].scatter(y_test, y_pred, alpha=0.2, s=6, color='#1976D2')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1)
axes[0].set_xlabel('Actual Power (kW)')
axes[0].set_ylabel('Predicted Power (kW)')
axes[0].set_title('Actual vs Predicted Power')

# Feature Importance
importances = model.feature_importances_
sorted_idx = np.argsort(importances)
axes[1].barh(
    np.array(features)[sorted_idx],
    importances[sorted_idx],
    color='#4CAF50'
)
axes[1].set_xlabel('Feature Importance')
axes[1].set_title('XGBoost Feature Importance')

plt.tight_layout()
plt.show()

## Module 8 — Performance Analysis

In [ ]:
# Predict expected power for the entire dataset
merged['expected_power'] = model.predict(merged[features].fillna(0))

# Power difference (expected - actual)
merged['power_diff'] = merged['expected_power'] - merged['total_power']

# Model-based Performance Ratio
merged['model_perf_ratio'] = np.where(
    merged['expected_power'] > 50,
    merged['total_power'] / merged['expected_power'],
    np.nan
)
merged['model_perf_ratio'] = merged['model_perf_ratio'].clip(0, 1.2)

# Loss percentage
merged['loss_pct'] = np.where(
    merged['expected_power'] > 50,
    (merged['power_diff'] / merged['expected_power']) * 100,
    0
)
merged['loss_pct'] = merged['loss_pct'].clip(0, 100)

print('Performance metrics computed.')
display(
    merged.loc[
        merged['irradiance'] > 300
    ].sample(10, random_state=42)[
        ['measured_on', 'total_power', 'expected_power',
         'model_perf_ratio', 'loss_pct']
    ].round(2)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Expected vs Actual Power (time series sample)
sample_day = merged[
    (merged['measured_on'].dt.date == merged['measured_on'].dt.date.mode()[0])
    & (merged['irradiance'] > 50)
]
axes[0].plot(sample_day['measured_on'], sample_day['total_power'],
             label='Actual', linewidth=1.2, color='#2196F3')
axes[0].plot(sample_day['measured_on'], sample_day['expected_power'],
             label='Expected', linewidth=1.2, color='#FF5722', linestyle='--')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Power (kW)')
axes[0].set_title('Actual vs Expected Power (Sample Day)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)

# Model-based PR trend
daily_mpr = merged.loc[
    merged['model_perf_ratio'].notna()
].groupby('date')['model_perf_ratio'].median()
axes[1].plot(daily_mpr.index, daily_mpr.values, linewidth=0.8, color='#009688')
axes[1].axhline(1.0, color='green', linestyle='--', alpha=0.5)
axes[1].axhline(0.8, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Median Model PR')
axes[1].set_title('Model-Based Performance Ratio Trend')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Module 9 — Cleaning Recommendation Engine

In [ ]:
def classify_condition(row):
    """
    Rule-based classification using model performance ratio,
    power difference, and estimated power loss.

    Returns (recommendation, reason).
    """
    pr       = row.get('model_perf_ratio', np.nan)
    loss     = row.get('loss_pct', 0)
    irr      = row.get('irradiance', 0)
    act_pwr  = row.get('total_power', 0)
    exp_pwr  = row.get('expected_power', 0)

    # --- Guard conditions ---
    if irr < 200:
        return 'Low Sunlight', 'Irradiance below 200 W/m² — night or overcast'
    if exp_pwr < 100:
        return 'Low Generation', 'Expected generation too low for analysis'
    if act_pwr < 50:
        return 'Investigate', 'Actual output near zero — possible equipment issue'

    # --- Core rules ---
    if pd.isna(pr):
        return 'Insufficient Data', 'Cannot compute performance ratio'
    if pr >= 0.95:
        return 'Healthy', 'Performance ratio within normal range'
    if pr >= 0.85:
        return 'Monitor', f'Mild degradation detected (PR={pr:.2f})'
    if pr >= 0.70:
        return 'Schedule Cleaning', f'Significant loss — PR={pr:.2f}, Loss={loss:.1f}%'
    return 'Clean Immediately', f'Severe loss — PR={pr:.2f}, Loss={loss:.1f}%'


results = merged.apply(classify_condition, axis=1, result_type='expand')
merged['recommendation'] = results[0]
merged['reason'] = results[1]

print('Recommendation Summary')
print(merged['recommendation'].value_counts().to_string())

In [ ]:
# Show sample recommendations for actionable conditions
actionable = merged[
    merged['recommendation'].isin(['Monitor', 'Schedule Cleaning', 'Clean Immediately'])
].sort_values('loss_pct', ascending=False)

display(
    actionable[
        ['measured_on', 'irradiance', 'total_power', 'expected_power',
         'model_perf_ratio', 'loss_pct', 'recommendation', 'reason']
    ].head(15).round(2)
)

In [ ]:
rec_counts = merged['recommendation'].value_counts()
colors = {
    'Healthy': '#4CAF50', 'Low Sunlight': '#9E9E9E',
    'Monitor': '#FFC107', 'Schedule Cleaning': '#FF9800',
    'Clean Immediately': '#F44336', 'Low Generation': '#B0BEC5',
    'Investigate': '#795548', 'Insufficient Data': '#BDBDBD'
}
bar_colors = [colors.get(r, '#607D8B') for r in rec_counts.index]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(rec_counts.index, rec_counts.values, color=bar_colors, edgecolor='white')
ax.set_xlabel('Number of Records')
ax.set_title('Cleaning Recommendation Distribution')
for i, v in enumerate(rec_counts.values):
    ax.text(v + max(rec_counts) * 0.01, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Module 10 — Operational Dashboard

In [ ]:
def build_dashboard(df):
    """
    Render a comprehensive Matplotlib-only dashboard showing
    KPIs, trends, and cleaning recommendations.
    """
    # --- Filter to meaningful operating conditions ---
    op = df[
        (df['irradiance'] > 200) &
        (df['expected_power'] > 100) &
        (df['total_power'] > 50)
    ].copy()

    # --- KPI values ---
    avg_power   = op['total_power'].mean()
    avg_irr     = op['irradiance'].mean()
    avg_pr      = op['model_perf_ratio'].mean()
    avg_loss    = op['loss_pct'].mean()
    rec_counts  = op['recommendation'].value_counts()
    n_cleaning  = rec_counts.get('Schedule Cleaning', 0) + rec_counts.get('Clean Immediately', 0)

    # --- Set up figure ---
    fig = plt.figure(figsize=(18, 16))
    fig.patch.set_facecolor('#FAFAFA')
    gs = gridspec.GridSpec(4, 2, hspace=0.4, wspace=0.3)

    # === Title ===
    fig.suptitle(
        'Solar Plant — Soiling Intelligence Dashboard',
        fontsize=18, fontweight='bold', y=0.98, color='#263238'
    )

    # === Row 0: KPI Cards ===
    ax_kpi = fig.add_subplot(gs[0, :])
    ax_kpi.axis('off')
    kpis = [
        ('Avg Power', f'{avg_power:,.0f} kW', '#1976D2'),
        ('Avg Irradiance', f'{avg_irr:,.0f} W/m²', '#FF9800'),
        ('Avg Perf. Ratio', f'{avg_pr:.2f}', '#4CAF50'),
        ('Avg Loss', f'{avg_loss:.1f}%', '#F44336'),
        ('Cleaning Alerts', f'{n_cleaning:,}', '#9C27B0'),
    ]
    for idx, (label, value, color) in enumerate(kpis):
        x_pos = 0.1 + idx * 0.18
        ax_kpi.text(
            x_pos, 0.65, value, fontsize=20, fontweight='bold',
            ha='center', va='center', color=color,
            transform=ax_kpi.transAxes
        )
        ax_kpi.text(
            x_pos, 0.25, label, fontsize=10, ha='center',
            va='center', color='#555555',
            transform=ax_kpi.transAxes
        )

    # === Row 1 Left: Power Generation Trend ===
    ax1 = fig.add_subplot(gs[1, 0])
    daily_pwr = op.groupby('date')['total_power'].mean()
    ax1.fill_between(range(len(daily_pwr)), daily_pwr.values, alpha=0.3, color='#1976D2')
    ax1.plot(range(len(daily_pwr)), daily_pwr.values, linewidth=0.8, color='#1976D2')
    ax1.set_title('Daily Power Generation Trend')
    ax1.set_ylabel('Avg Power (kW)')
    ax1.set_xlabel('Day Index')

    # === Row 1 Right: Actual vs Expected Power ===
    ax2 = fig.add_subplot(gs[1, 1])
    sample = op.sample(min(3000, len(op)), random_state=42)
    ax2.scatter(
        sample['expected_power'], sample['total_power'],
        alpha=0.2, s=6, color='#2196F3'
    )
    lims = [
        min(sample['expected_power'].min(), sample['total_power'].min()),
        max(sample['expected_power'].max(), sample['total_power'].max())
    ]
    ax2.plot(lims, lims, 'r--', linewidth=1)
    ax2.set_title('Actual vs Expected Power')
    ax2.set_xlabel('Expected (kW)')
    ax2.set_ylabel('Actual (kW)')

    # === Row 2 Left: Performance Ratio Trend ===
    ax3 = fig.add_subplot(gs[2, 0])
    daily_pr = op.groupby('date')['model_perf_ratio'].median()
    ax3.plot(range(len(daily_pr)), daily_pr.values, linewidth=0.8, color='#673AB7')
    ax3.axhline(1.0, color='green', linestyle='--', alpha=0.5, linewidth=0.8)
    ax3.axhline(0.8, color='red', linestyle='--', alpha=0.5, linewidth=0.8)
    ax3.set_title('Daily Performance Ratio Trend')
    ax3.set_ylabel('Median PR')
    ax3.set_xlabel('Day Index')

    # === Row 2 Right: Recommendation Distribution ===
    ax4 = fig.add_subplot(gs[2, 1])
    op_rec = op['recommendation'].value_counts()
    pie_colors = {
        'Healthy': '#4CAF50', 'Monitor': '#FFC107',
        'Schedule Cleaning': '#FF9800', 'Clean Immediately': '#F44336',
        'Low Sunlight': '#9E9E9E', 'Low Generation': '#B0BEC5',
        'Investigate': '#795548', 'Insufficient Data': '#BDBDBD'
    }
    clrs = [pie_colors.get(r, '#607D8B') for r in op_rec.index]
    wedges, texts, autotexts = ax4.pie(
        op_rec.values, labels=op_rec.index, autopct='%1.1f%%',
        colors=clrs, textprops={'fontsize': 8},
        pctdistance=0.80, startangle=140
    )
    for at in autotexts:
        at.set_fontsize(7)
    ax4.set_title('Cleaning Recommendation Distribution')

    # === Row 3: Top Cleaning Candidates Table ===
    ax5 = fig.add_subplot(gs[3, :])
    ax5.axis('off')
    ax5.set_title('Top Cleaning Candidates', fontsize=13, fontweight='bold', pad=10)

    top_candidates = op[
        op['recommendation'].isin(['Schedule Cleaning', 'Clean Immediately', 'Monitor'])
    ].nlargest(10, 'loss_pct')[
        ['measured_on', 'total_power', 'expected_power',
         'model_perf_ratio', 'loss_pct', 'recommendation', 'reason']
    ].copy()
    top_candidates.columns = [
        'Timestamp', 'Actual (kW)', 'Expected (kW)',
        'Perf. Ratio', 'Loss %', 'Action', 'Reason'
    ]
    top_candidates = top_candidates.round(2)
    top_candidates['Timestamp'] = top_candidates['Timestamp'].dt.strftime('%Y-%m-%d %H:%M')

    if len(top_candidates) > 0:
        table = ax5.table(
            cellText=top_candidates.values,
            colLabels=top_candidates.columns,
            loc='center',
            cellLoc='center'
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1.0, 1.4)
        # Style header
        for j in range(len(top_candidates.columns)):
            table[0, j].set_facecolor('#263238')
            table[0, j].set_text_props(color='white', fontweight='bold')
        # Alternate row colours
        for i in range(1, len(top_candidates) + 1):
            bg = '#F5F5F5' if i % 2 == 0 else 'white'
            for j in range(len(top_candidates.columns)):
                table[i, j].set_facecolor(bg)
    else:
        ax5.text(0.5, 0.4, 'No cleaning candidates found.',
                 ha='center', va='center', fontsize=12, color='#4CAF50')

    plt.show()
    print('Dashboard rendered successfully.')


build_dashboard(merged)

## Summary

In [ ]:
print('=' * 65)
print('  SOILING INTELLIGENCE PoC — FINAL SUMMARY')
print('=' * 65)
print(f'  Records processed        : {len(merged):,}')
print(f'  ML Model                 : XGBoost Regressor')
print(f'  R² Score                 : {r2:.4f}')
print(f'  MAE                      : {mae:.2f} kW')
print(f'  RMSE                     : {rmse:.2f} kW')
print(f'  Avg Performance Ratio    : {merged["model_perf_ratio"].mean():.3f}')
print(f'  Avg Power Loss           : {merged["loss_pct"].mean():.2f}%')
print('-' * 65)

rec_summary = merged['recommendation'].value_counts()
for status, count in rec_summary.items():
    print(f'  {status:<25}: {count:>8,}')

print('=' * 65)
print('  PoC-2 execution complete.')